# PPP Fraud Intelligence System: A Multi-Agent RAG and SQL Analytics Platform

## Installation of Libraries

In [17]:
!pip install -q \
    langchain \
    langchain-anthropic \
    langchain-community \
    langchain-groq \
    sqlalchemy \
    pandas \
    faiss-cpu \
    pypdf \
    sentence-transformers \
    tqdm \
    groq

## Mount Drive & Locate the CSV

In [2]:
from google.colab import drive, files
import os

drive.mount('/content/drive')

# ── Option A: Point to a file already in Google Drive ────────────────────────
CSV_PATH = "/content/150plus_n.csv"

# ── Option B: Upload directly (slower for 430MB, but works) ──────────────────
if not CSV_PATH or not os.path.exists(CSV_PATH):
    print("CSV not found at CSV_PATH. Uploading manually...")
    uploaded = files.upload()
    CSV_PATH = list(uploaded.keys())[0]

print(f"Using CSV: {CSV_PATH}")
print(f"File size: {os.path.getsize(CSV_PATH) / 1e6:.1f} MB")

Mounted at /content/drive
Using CSV: /content/150plus_n.csv
File size: 165.7 MB


In [3]:
from google.colab import files as colab_files

print("Upload your fraud reference PDFs (select all at once):")
print("  - gao-23-105331.pdf")
print("  - IsFraudContagious_SocialConnections_preview.pdf")
print("  - FAQ_PPP_for_Borrowers_and_Lenders.pdf")
print("  - PPP-Borrower-Application-Form.pdf")
print("  - PPP-Second-Draw-Borrower-Application-Form.pdf")
print()

uploaded_pdfs = colab_files.upload()
PDF_PATHS = list(uploaded_pdfs.keys())
print(f"\nUploaded {len(PDF_PATHS)} PDFs:")
for p in PDF_PATHS:
    print(f"  {p}")

Upload your fraud reference PDFs (select all at once):
  - gao-23-105331.pdf
  - IsFraudContagious_SocialConnections_preview.pdf
  - FAQ_PPP_for_Borrowers_and_Lenders.pdf
  - PPP-Borrower-Application-Form.pdf
  - PPP-Second-Draw-Borrower-Application-Form.pdf



Saving FAQ PPP for Borrowers and Lenders Questions 1-73 (FINAL 5-9-24) 508.pdf to FAQ PPP for Borrowers and Lenders Questions 1-73 (FINAL 5-9-24) 508 (1).pdf
Saving gao-23-105331.pdf to gao-23-105331 (1).pdf
Saving IsFraudContagious_SocialConnections_preview.pdf to IsFraudContagious_SocialConnections_preview (1).pdf
Saving PPP-Second-Draw-Borrower-Application-Form.pdf to PPP-Second-Draw-Borrower-Application-Form (1).pdf
Saving PPP-Borrower-Application-Form.pdf to PPP-Borrower-Application-Form (1).pdf

Uploaded 5 PDFs:
  FAQ PPP for Borrowers and Lenders Questions 1-73 (FINAL 5-9-24) 508 (1).pdf
  gao-23-105331 (1).pdf
  IsFraudContagious_SocialConnections_preview (1).pdf
  PPP-Second-Draw-Borrower-Application-Form (1).pdf
  PPP-Borrower-Application-Form (1).pdf


## Set API Keys

In [5]:
import os
from google.colab import userdata

try:
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    print("API key loaded from Colab Secrets.")
except Exception:
    import getpass
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")
    print("API key set manually.")

API key loaded from Colab Secrets.


## Convert CSV → SQLite

In [6]:
import pandas as pd
from sqlalchemy import create_engine
from tqdm import tqdm

os.makedirs("db", exist_ok=True)
DATABASE_PATH = "./db/ppp.db"
TABLE_NAME    = "ppp_loans"
CHUNK_SIZE    = 50_000

engine = create_engine(f"sqlite:///{DATABASE_PATH}")

print(f"Converting CSV → SQLite ({DATABASE_PATH})...")
print("This may take 2–4 minutes for a 430MB file.\n")

chunk_iter = pd.read_csv(
    CSV_PATH,
    chunksize=CHUNK_SIZE,
    dtype=str,           # read everything as string first to avoid type errors
    low_memory=False
)

total_rows = 0
for i, chunk in enumerate(tqdm(chunk_iter, desc="Writing chunks")):
    # Coerce numeric columns after load to avoid SQLite type issues
    numeric_cols = [
        "InitialApprovalAmount", "CurrentApprovalAmount", "UndisbursedAmount",
        "ForgivenessAmount", "JobsReported", "Term", "SBAGuarantyPercentage",
        "UTILITIES_PROCEED", "PAYROLL_PROCEED", "MORTGAGE_INTEREST_PROCEED",
        "RENT_PROCEED", "REFINANCE_EIDL_PROCEED", "HEALTH_CARE_PROCEED",
        "DEBT_INTEREST_PROCEED"
    ]
    for col in numeric_cols:
        if col in chunk.columns:
            chunk[col] = pd.to_numeric(chunk[col], errors="coerce")

    chunk.to_sql(
        TABLE_NAME,
        con=engine,
        if_exists="replace" if i == 0 else "append",
        index=False
    )
    total_rows += len(chunk)

print(f"\nDone. {total_rows:,} rows written to table '{TABLE_NAME}'.")

# Quick sanity check
sample = pd.read_sql(f"SELECT COUNT(*) as cnt FROM {TABLE_NAME}", engine)
print(f"SQLite row count: {sample['cnt'][0]:,}")

Converting CSV → SQLite (./db/ppp.db)...
This may take 2–4 minutes for a 430MB file.



Writing chunks: 20it [01:07,  3.37s/it]


Done. 968,525 rows written to table 'ppp_loans'.
SQLite row count: 968,525


## Build the RAG Index

In [8]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

print("Loading and chunking PDFs...")

all_docs = []
for pdf_path in PDF_PATHS:
    loader = PyPDFLoader(pdf_path)
    docs   = loader.load()
    # Tag each chunk with its source filename for attribution
    for doc in docs:
        doc.metadata["source"] = os.path.basename(pdf_path)
    all_docs.extend(docs)
    print(f"  Loaded {len(docs)} pages from {os.path.basename(pdf_path)}")

print(f"\nTotal pages loaded: {len(all_docs)}")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " "]
)
chunks = splitter.split_documents(all_docs)
print(f"Total chunks after splitting: {len(chunks)}")

print("\nBuilding FAISS vector index (runs locally, no API key needed)...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever   = vectorstore.as_retriever(search_kwargs={"k": 5})

print("RAG index ready.")

Loading and chunking PDFs...
  Loaded 34 pages from FAQ PPP for Borrowers and Lenders Questions 1-73 (FINAL 5-9-24) 508 (1).pdf
  Loaded 154 pages from gao-23-105331 (1).pdf
  Loaded 110 pages from IsFraudContagious_SocialConnections_preview (1).pdf
  Loaded 7 pages from PPP-Second-Draw-Borrower-Application-Form (1).pdf
  Loaded 7 pages from PPP-Borrower-Application-Form (1).pdf

Total pages loaded: 312
Total chunks after splitting: 1088

Building FAISS vector index (runs locally, no API key needed)...


/tmp/ipykernel_11540/2260211090.py:29: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

RAG index ready.


## Build the SQL Agent

In [9]:
from langchain_groq import ChatGroq
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit, create_sql_agent

PPP_DATA_DICTIONARY = """
Table: ppp_loans
Key columns for fraud analysis:
  - LoanNumber               Unique loan identifier
  - DateApproved             Date loan was funded
  - ProcessingMethod         PPP (first draw) or PPS (second draw)
  - BorrowerName             Business name
  - BorrowerAddress/City/State/Zip  Borrower location
  - LoanStatus               Current loan status
  - InitialApprovalAmount    Original loan amount (numeric)
  - CurrentApprovalAmount    Current loan amount (numeric)
  - ForgivenessAmount        Amount forgiven (numeric; NULL if not forgiven)
  - ForgivenessDate          Date forgiveness was paid
  - UndisbursedAmount        Amount not yet disbursed
  - JobsReported             Number of employees reported (numeric)
  - NAICSCode                6-digit industry code
  - BusinessType             Legal entity type (Sole Proprietorship, LLC, etc.)
  - BusinessAgeDescription   Age of business at time of application
  - RuralUrbanIndicator      R = Rural, U = Urban
  - HubzoneIndicator         Y/N
  - LMIIndicator             Low-to-moderate income area Y/N
  - NonProfit                'Yes' if nonprofit
  - Gender / Race / Ethnicity / Veteran  Demographic fields (optional, may be blank)
  - OriginatingLender        Name of originating lender
  - ServicingLenderName      Name of servicing lender
  - FranchiseName            Franchise name if applicable
  - PAYROLL_PROCEED          Payroll use flag
  - UTILITIES_PROCEED        Utilities use flag
  - MORTGAGE_INTEREST_PROCEED  Mortgage interest use flag
  - RENT_PROCEED             Rent use flag
  - REFINANCE_EIDL_PROCEED   EIDL refinance use flag
  - HEALTH_CARE_PROCEED      Health care use flag
  - DEBT_INTEREST_PROCEED    Debt interest use flag
  - Term                     Loan maturity in months
  - SBAGuarantyPercentage    SBA guarantee %
  - SBAOfficeCode            SBA originating office code
  - CD                       Congressional district

Fraud-relevant derived metrics to compute in SQL:
  - loan_per_job  = InitialApprovalAmount / NULLIF(JobsReported, 0)
  - unforgiven_ratio = (InitialApprovalAmount - COALESCE(ForgivenessAmount,0)) / InitialApprovalAmount
  - is_round_amount: InitialApprovalAmount % 10000 = 0
"""

SQL_AGENT_PREFIX = f"""
You are a PPP loan fraud analyst assistant with access to a SQLite database
containing SBA Paycheck Protection Program loan records.

## Dataset reference:
{PPP_DATA_DICTIONARY}

## Instructions:
- Write syntactically correct SQLite queries, execute them, and return
  clear plain-English answers based solely on the results.
- ALWAYS limit results to at most {{top_k}} rows unless the user specifies more.
- Only SELECT the columns needed. Never use SELECT *.
- Double-check your query before running it. Retry with a corrected query on error.
- DO NOT make DML statements (INSERT, UPDATE, DELETE, DROP).
- DO NOT make up answers — use only what your SQL returns.
- If the question is about schema or column meanings, answer from the data
  dictionary above without running a query.
- Always end with an "Explanation:" section describing the query and findings.
- Fraud indicator queries to consider:
    * Loans with JobsReported = 0 or 1 but InitialApprovalAmount > 50000
    * InitialApprovalAmount exactly divisible by 10000 (round number flag)
    * InitialApprovalAmount > 10x average for same NAICSCode
    * ForgivenessAmount IS NULL for loans with LoanStatus = 'Paid in Full'
    * Multiple loans to same BorrowerName or BorrowerAddress
    * BusinessAgeDescription = 'Unanswered' or 'Not Applicable' (potential shell)

## Tools:
"""

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, max_tokens=2048)

db      = SQLDatabase.from_uri(f"sqlite:///{DATABASE_PATH}")
toolkit = SQLDatabaseToolkit(db=db, llm=llm)

sql_agent = create_sql_agent(
    prefix=SQL_AGENT_PREFIX,
    llm=llm,
    toolkit=toolkit,
    top_k=30,
    verbose=True,
    agent_executor_kwargs={"handle_parsing_errors": True},
)

print("SQL agent ready.")

SQL agent ready.


## Build the RAG Chain + Router

In [21]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from groq import Groq

# ── RAG prompt ────────────────────────────────────────────────────────────────
RAG_PROMPT = PromptTemplate.from_template("""You are a PPP fraud analysis assistant.
Answer the question using ONLY the context below. Always cite which document your
answer comes from (e.g., "According to the GAO report..." or "Per the SBA FAQ...").
If the context doesn't contain the answer, say so clearly rather than guessing.
Format your answer in plain English suitable for a fraud analyst or researcher.

Context:
{context}

Question: {question}

Answer:""")

# ── RAG chain (LCEL) ──────────────────────────────────────────────────────────
rag_llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, max_tokens=2048)

def format_docs(docs):
    return "\n\n".join(
        f"[{doc.metadata.get('source', 'unknown')}]\n{doc.page_content}"
        for doc in docs
    )

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | RAG_PROMPT
    | rag_llm
    | StrOutputParser()
)

# ── Router prompt ─────────────────────────────────────────────────────────────
ROUTER_PROMPT = """You are a query router for a PPP loan fraud analysis system.
Classify the user's question into exactly one of two categories:

  SQL   - The question requires querying loan records, statistics, counts,
          specific borrowers, lenders, states, amounts, or any numerical
          analysis of the dataset. Examples: "how many loans...", "which
          state had the most...", "show me loans where...", "find borrowers
          with zero jobs reported".

  RAG   - The question asks about fraud schemes, fraud indicators, regulatory
          rules, eligibility requirements, what constitutes fraud, how fraud
          spreads, GAO findings, or any conceptual / policy question that
          would be answered from reference documents. Examples: "what are
          common fraud schemes", "what does the GAO say about...", "what
          disqualifies a borrower", "how is fraud defined".

Respond with ONLY the single word: SQL or RAG"""

# ── Router client + function ──────────────────────────────────────────────────
router_client = Groq()

def route_question(question: str) -> str:
    resp = router_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        max_tokens=10,
        messages=[
            {"role": "system", "content": ROUTER_PROMPT},
            {"role": "user",   "content": question}
        ]
    )
    return resp.choices[0].message.content.strip().upper()

print("RAG chain and router ready.")

RAG chain and router ready.


## Conversation Memory + Interactive Chat Loop

In [22]:
chat_history = []   # list of (question, answer, route) tuples

def ask(question: str, context_turns: int = 3) -> tuple[str, str]:
    route = route_question(question)

    if chat_history:
        recent = chat_history[-context_turns:]
        history_block = "Previous conversation context:\n"
        for q, a, _ in recent:
            history_block += f"Q: {q}\nA: {a}\n\n"
        augmented = f"{history_block}Current question: {question}"
    else:
        augmented = question

    if route == "SQL":
        result = sql_agent.invoke({"input": augmented})
        answer = result["output"]
    else:  # RAG
        retrieved_docs = retriever.invoke(augmented)
        answer = rag_chain.invoke(augmented)
        sources = set(doc.metadata.get("source", "unknown") for doc in retrieved_docs)
        if sources:
            answer += f"\n\n📄 Sources: {', '.join(sorted(sources))}"

    chat_history.append((question, answer, route))
    return answer, route

def reset_history():
    chat_history.clear()
    print("Conversation history cleared.")

# ── Chat loop ─────────────────────────────────────────────────────────────────
print("=" * 65)
print("  PPP Loan Fraud Analysis Chatbot")
print("=" * 65)
print("Type 'exit' to quit | 'reset' to clear history | 'history' to review")
print()
print("SQL questions (loan data):")
print("  • Which states had the highest average loan per job reported?")
print("  • How many loans had JobsReported = 0 but amount over $50,000?")
print("  • Find loans with round amounts divisible by $10,000 and no forgiveness.")
print("  • Which lenders originated the most loans flagged as sole proprietors?")
print()
print("RAG questions (fraud knowledge):")
print("  • What are the most common PPP fraud schemes per the GAO report?")
print("  • What fraud indicators does the GAO identify in its data analysis?")
print("  • How did fraud spread geographically according to researchers?")
print("  • What disqualifies a borrower from receiving a PPP loan?")
print("=" * 65)

while True:
    try:
        question = input("\nYou: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\nGoodbye!")
        break

    if not question:
        continue
    if question.lower() in ("exit", "quit", "q"):
        print("Goodbye!")
        break
    if question.lower() == "reset":
        reset_history()
        continue
    if question.lower() == "history":
        if not chat_history:
            print("No history yet.")
        for i, (q, a, r) in enumerate(chat_history, 1):
            print(f"\n[{i}] [{r}] Q: {q}\nA: {a[:200]}...")
        continue

    try:
        answer, route = ask(question)
        print(f"\n[Routed to: {route}]")
        print(f"\nAgent: {answer}")
        print("─" * 65)
    except Exception as e:
        print(f"\n[Error] {e}")
        print("Try rephrasing, or type 'reset' and try again.")

  PPP Loan Fraud Analysis Chatbot
Type 'exit' to quit | 'reset' to clear history | 'history' to review

SQL questions (loan data):
  • Which states had the highest average loan per job reported?
  • How many loans had JobsReported = 0 but amount over $50,000?
  • Find loans with round amounts divisible by $10,000 and no forgiveness.
  • Which lenders originated the most loans flagged as sole proprietors?

RAG questions (fraud knowledge):
  • What are the most common PPP fraud schemes per the GAO report?
  • What fraud indicators does the GAO identify in its data analysis?
  • How did fraud spread geographically according to researchers?
  • What disqualifies a borrower from receiving a PPP loan?

You: Which states had the highest average loan per job reported?


> Entering new SQL Agent Executor chain...
Thought: I should look at the tables in the database to see what I can query.  Then I should query the schema of the most relevant tables.
Action: sql_db_list_tables
Action Input: ppp_

## Chat interface with Gradio

In [23]:
!pip install -q gradio

In [ ]:
import gradio as gr

def chat(user_message, history):
    try:
        answer, route = ask(user_message)
        label = f"[Routed to: {route}]"
        return history + [[user_message, f"{label}\n\n{answer}"]]
    except Exception as e:
        return history + [[user_message, f"[Error] {str(e)}"]]

def clear():
    reset_history()
    return []

with gr.Blocks(title="PPP Fraud Analysis Chatbot") as demo:
    gr.Markdown("## PPP Loan Fraud Analysis Chatbot")
    gr.Markdown("Ask data questions (SQL) or fraud knowledge questions (RAG) — the router handles the rest.")

    chatbot = gr.Chatbot(
        value=[],
        height=500,
        label="Conversation"
    )
    with gr.Row():
        msg = gr.Textbox(
            placeholder="e.g. 'Which states had the most loans with zero jobs reported?' or 'What fraud schemes does the GAO identify?'",
            label="Your question",
            scale=4
        )
        submit_btn = gr.Button("Send", scale=1, variant="primary")

    with gr.Row():
        clear_btn = gr.Button("Clear conversation")

    gr.Examples(
        examples=[
            "How many loans had JobsReported = 0 but InitialApprovalAmount over $50,000?",
            "Which states had the highest average loan per job reported?",
            "What are the most common PPP fraud schemes per the GAO report?",
            "What disqualifies a borrower from receiving a PPP loan?",
            "Find loans with round amounts divisible by $10,000 and no forgiveness.",
            "How did fraud spread geographically according to researchers?",
        ],
        inputs=msg
    )

    submit_btn.click(chat, inputs=[msg, chatbot], outputs=chatbot)
    msg.submit(chat, inputs=[msg, chatbot], outputs=chatbot)
    clear_btn.click(clear, outputs=chatbot)

demo.launch(share=True, debug=True)

/tmp/ipykernel_11540/1212139573.py:19: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipykernel_11540/1212139573.py:19: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c6aff28d8bcad3de80.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)




> Entering new SQL Agent Executor chain...
